# Image Forgery Detection â€” Dual-Input EfficientNetB3

Two EfficientNetB3 backbones (raw RGB + ELA residuals) fused via cross-attention (product + difference).
Includes JPEG simulation (format confound fix), SBI augmentation, TTA, mixed precision, and OverfitGuard.

**Platform:** Kaggle (P100) or Colab (T4) â€” auto-detected.
**Run time:** ~30-40 min on Kaggle P100 (first run includes cache precomputation)

In [ ]:
# === PLATFORM + DATASET SETUP ===
# Toggle: 'defacto' (247K, primary) | 'casia' (12K, fallback)
DATASET_MODE = 'defacto'

import os
from pathlib import Path
IN_COLAB = 'COLAB_GPU' in os.environ
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
IS_KAGGLE = IN_KAGGLE

# Platform-specific dataset detection
if IN_KAGGLE:
    import kagglehub
    if DATASET_MODE == 'defacto':
        # Actual DEFACTO layout:
        #   splicing:  splicing_1_img/ .. splicing_7_img/  (images),  *_annotations/ (masks)
        #   copymove:  copymove_img/img/  (images),  copymove_annotations/ (masks)
        #   coco:      coco2017/train2017/
        def _find_splice_root(root):
            """Return dir that contains splicing_*_img/ subdirs."""
            if any(root.glob('splicing_*_img')): return root
            for d in root.rglob('splicing_*_img'):
                if d.is_dir(): return d.parent
            return None
        def _find_cm_root(root):
            """Return dir that contains copymove_img/ subdir."""
            if (root / 'copymove_img').exists(): return root
            for d in root.rglob('copymove_img'):
                if d.is_dir(): return d.parent
            return None
        def _find_coco_train(root):
            """Return dir that has train2017/ as child."""
            if (root / 'coco2017' / 'train2017').exists(): return root / 'coco2017'
            if (root / 'train2017').exists(): return root
            for d in root.rglob('train2017'):
                if d.is_dir(): return d.parent
            return None
        # Try mounted paths first (covers common Kaggle naming variants)
        _spl = _cm = _coco = None
        for _mnt in ['/kaggle/input/defacto-splicing', '/kaggle/input/defactosplicing']:
            if Path(_mnt).exists():
                _spl = _find_splice_root(Path(_mnt)); break
        for _mnt in ['/kaggle/input/defacto-copymove', '/kaggle/input/defactocopymove']:
            if Path(_mnt).exists():
                _cm = _find_cm_root(Path(_mnt)); break
        for _mnt in ['/kaggle/input/coco-2017-dataset', '/kaggle/input/coco2017']:
            if Path(_mnt).exists():
                _coco = _find_coco_train(Path(_mnt)); break
        if not (_spl and _cm and _coco):
            print('Datasets not fully mounted. Downloading via Kaggle API (one-time)...')
            _dl = kagglehub.dataset_download
            _splice_raw = Path(_dl('defactodataset/defactosplicing'))
            _cm_raw     = Path(_dl('defactodataset/defactocopymove'))
            _coco_raw   = Path(_dl('awsaf49/coco-2017-dataset'))
            for name, p in [('splice', _splice_raw), ('cm', _cm_raw), ('coco', _coco_raw)]:
                print(f'  {name}: {p}')
                if p.exists():
                    for d in sorted(p.iterdir())[:12]:
                        print(f'    {d.name}/' if d.is_dir() else f'    {d.name}')
            if not _spl: _spl = _find_splice_root(_splice_raw)
            if not _cm:  _cm  = _find_cm_root(_cm_raw)
            if not _coco: _coco = _find_coco_train(_coco_raw)
        if not (_spl and _cm and _coco):
            raise RuntimeError('Could not locate DEFACTO directories. Check cell output above.')
        DEFACTO_SPLICE_ROOT = _spl
        DEFACTO_CM_ROOT     = _cm
        COCO_ROOT           = _coco
        # List actual image dirs found
        _spl_dirs = sorted(_spl.glob('splicing_*_img'))
        _cm_dir   = _cm / 'copymove_img' / 'img'
        _coco_dir = _coco / 'train2017'
        print(f'Splicing image dirs ({len(_spl_dirs)}): {[d.name for d in _spl_dirs]}')
        print(f'Copy-move image dir: {_cm_dir} (exists={_cm_dir.exists()})')
        print(f'COCO train2017: {_coco_dir} (exists={_coco_dir.exists()})')
    if DATASET_MODE == 'casia':
        _mounted_casia = Path('/kaggle/input/casia-20-image-tampering-detection-dataset')
        if _mounted_casia.exists():
            CASIA_DS = _mounted_casia
        else:
            candidates = sorted(Path('/kaggle/input').glob('*/Au'))
            if candidates:
                CASIA_DS = candidates[0].parent
            else:
                print('CASIA not mounted. Downloading via Kaggle API...')
                CASIA_DS = Path(kagglehub.dataset_download('divg07/casia-20-image-tampering-detection-dataset'))
                # Find Au subdir
                _au = list(CASIA_DS.rglob('Au'))
                if _au: CASIA_DS = _au[0].parent
        DATASET_DIR = str(CASIA_DS)
        print(f'CASIA: {DATASET_DIR}')
elif IN_COLAB:
    import subprocess, sys, importlib
    _pkgs = {'scikit-learn': 'sklearn', 'Pillow': 'PIL', 'opencv-python-headless': 'cv2', 'kagglehub': 'kagglehub'}
    for pkg, mod in _pkgs.items():
        try:
            importlib.import_module(mod)
        except ImportError:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    if DATASET_MODE == 'defacto':
        import kagglehub
        try:
            defacto_splice_dir = kagglehub.dataset_download('defactodataset/defactosplicing')
            defacto_cm_dir = kagglehub.dataset_download('defactodataset/defactocopymove')
            coco_dir = kagglehub.dataset_download('awsaf49/coco-2017-dataset')
            def _find_splice_root(root):
                if any(Path(root).glob('splicing_*_img')): return Path(root)
                for d in Path(root).rglob('splicing_*_img'):
                    if d.is_dir(): return d.parent
                return None
            def _find_cm_root(root):
                r = Path(root)
                if (r / 'copymove_img').exists(): return r
                for d in r.rglob('copymove_img'):
                    if d.is_dir(): return d.parent
                return None
            def _find_coco_train(root):
                r = Path(root)
                if (r / 'coco2017' / 'train2017').exists(): return r / 'coco2017'
                if (r / 'train2017').exists(): return r
                for d in r.rglob('train2017'):
                    if d.is_dir(): return d.parent
                return None
            DEFACTO_SPLICE_ROOT = _find_splice_root(defacto_splice_dir)
            DEFACTO_CM_ROOT     = _find_cm_root(defacto_cm_dir)
            COCO_ROOT           = _find_coco_train(coco_dir)
            print(f'DEFACTO splicing root: {DEFACTO_SPLICE_ROOT}')
            print(f'DEFACTO copymove root: {DEFACTO_CM_ROOT}')
            print(f'COCO root: {COCO_ROOT}')
        except Exception as e:
            print(f'DEFACTO download failed: {e}')
            DATASET_MODE = 'casia'
    if DATASET_MODE == 'casia':
        import kagglehub
        try:
            DATASET_DIR = kagglehub.dataset_download('divg07/casia-20-image-tampering-detection-dataset')
            for child in Path(DATASET_DIR).rglob('*'):
                if child.name == 'Au' and child.is_dir():
                    DATASET_DIR = str(child.parent); break
            print(f'CASIA: {DATASET_DIR}')
        except Exception as e:
            print(f'CASIA download failed: {e}')
            DATASET_DIR = None
else:
    DATASET_DIR = None

print(f'Mode: {DATASET_MODE} | Platform: {"Kaggle" if IN_KAGGLE else "Colab" if IN_COLAB else "Local"}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import json, gc, math
from pathlib import Path
from PIL import Image, ImageChops, ImageEnhance
import cv2
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import random

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

print(f'TensorFlow: {tf.__version__}')
tf.keras.mixed_precision.set_global_policy('mixed_float16')
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass
print(f'Mixed precision: {tf.keras.mixed_precision.global_policy().name}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')
import threading, time
def _keepalive():
    while True:
        time.sleep(60)
        print('.', end='', flush=True)
if IN_COLAB or IN_KAGGLE:
    t = threading.Thread(target=_keepalive, daemon=True)
    t.start()
    print('Keepalive thread started.')

In [ ]:
# Validate dataset paths
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
ELA_QUALITY = 91

if DATASET_MODE == 'defacto':
    for name in ['DEFACTO_SPLICE_ROOT', 'DEFACTO_CM_ROOT', 'COCO_ROOT']:
        if name not in dir() or not isinstance(eval(name), Path):
            raise RuntimeError(f'{name} not defined. Download may have failed. Check cell 2 output.')
    for name, p in [('DEFACTO_SPLICE_ROOT', DEFACTO_SPLICE_ROOT), ('DEFACTO_CM_ROOT', DEFACTO_CM_ROOT), ('COCO_ROOT', COCO_ROOT)]:
        if not p.exists():
            print(f'{name} NOT FOUND: {p}')
            if p.parent.exists():
                print(f'  Contents of {p.parent}:')
                for d in sorted(p.parent.iterdir())[:15]:
                    print(f'    {d.name}/' if d.is_dir() else f'    {d.name}')
    assert DEFACTO_SPLICE_ROOT.exists(), f'Missing: {DEFACTO_SPLICE_ROOT}'
    assert DEFACTO_CM_ROOT.exists(),      f'Missing: {DEFACTO_CM_ROOT}'
    assert COCO_ROOT.exists(),            f'Missing: {COCO_ROOT}'
    _spl_dirs = sorted(DEFACTO_SPLICE_ROOT.glob('splicing_*_img'))
    assert len(_spl_dirs) > 0, f'No splicing_*_img dirs in {DEFACTO_SPLICE_ROOT}'
    assert (DEFACTO_CM_ROOT / 'copymove_img').exists(), f'Missing copymove_img in {DEFACTO_CM_ROOT}'
    assert (COCO_ROOT / 'train2017').exists(),          f'Missing train2017 in {COCO_ROOT}'
    print(f'Splicing dirs ({len(_spl_dirs)}): {[d.name for d in _spl_dirs]}')
    print(f'Copy-move root: {DEFACTO_CM_ROOT}')
    print(f'COCO root: {COCO_ROOT}')
elif DATASET_MODE == 'casia':
    DATA_ROOT = Path(DATASET_DIR) if DATASET_DIR else None
    if not DATA_ROOT:
        candidates = [Path.cwd() / 'dataset', Path.cwd().parent / 'dataset']
        DATA_ROOT = next((p for p in candidates if p.exists()), None)
    if not DATA_ROOT or not DATA_ROOT.exists():
        raise RuntimeError(
            'CASIA dataset not found. On Kaggle, add: divg07/casia-20-image-tampering-detection-dataset\n'
            'Or switch DATASET_MODE to defacto and add DEFACTO datasets.'
        )
    if not (DATA_ROOT / 'Au').exists() or not (DATA_ROOT / 'Tp').exists():
        raise RuntimeError(f'Missing Au/ or Tp/ folders in {DATA_ROOT}')
    print(f'CASIA: {DATA_ROOT}')

In [ ]:
import io
def compute_ela(image_path, quality=ELA_QUALITY, target_size=IMG_SIZE, pil_image=None):
    img = pil_image if pil_image is not None else Image.open(image_path).convert('RGB')
    buf = io.BytesIO()
    img.save(buf, 'JPEG', quality=quality)
    buf.seek(0)
    compressed = Image.open(buf)
    ela = ImageChops.difference(img, compressed)
    extrema = ela.getextrema()
    max_diff = max(ex[1] for ex in extrema)
    if max_diff == 0: max_diff = 1
    ela = ImageEnhance.Brightness(ela).enhance(255.0 / max_diff)
    return ela.resize(target_size, Image.LANCZOS)


In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}

def collect_paths(d):
    return sorted([str(p) for p in Path(d).rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS])

if DATASET_MODE == 'defacto':
    # DEFACTO: splicing + copy-move = tampered (0), COCO = authentic (1)
    forged_splice = []
    for d in sorted(DEFACTO_SPLICE_ROOT.glob('splicing_*_img')):
        forged_splice.extend(collect_paths(d))
    forged_cm = collect_paths(DEFACTO_CM_ROOT / 'copymove_img' / 'img')
    auth_paths = collect_paths(COCO_ROOT / 'train2017')
    forged_paths = forged_splice + forged_cm
    print(f'Forged splicing: {len(forged_splice)} | copy-move: {len(forged_cm)}')
    print(f'Authentic (COCO): {len(auth_paths)}')
    all_paths  = forged_paths + auth_paths
    all_labels = [0]*len(forged_paths) + [1]*len(auth_paths)
else:
    # CASIA v2
    au_paths = collect_paths(DATA_ROOT / 'Au')
    tp_paths = collect_paths(DATA_ROOT / 'Tp')
    print(f'CASIA: {len(au_paths)} authentic, {len(tp_paths)} tampered')
    all_paths  = au_paths + tp_paths
    all_labels = [1]*len(au_paths) + [0]*len(tp_paths)

print(f'Total: {len(all_paths)} ({sum(all_labels)} authentic, {len(all_labels)-sum(all_labels)} tampered)')

In [ ]:
# 70/15/15 stratified split (paths only â€” no images loaded yet)
paths_tmp, paths_test, labels_tmp, y_test = train_test_split(
    all_paths, all_labels, test_size=0.15, random_state=SEED, stratify=all_labels)
paths_train, paths_val, y_train, y_val = train_test_split(
    paths_tmp, labels_tmp, test_size=0.15/(1-0.15), random_state=SEED, stratify=labels_tmp)
y_val = np.array(y_val, dtype=np.int32)
y_test = np.array(y_test, dtype=np.int32)
y_train = np.array(y_train, dtype=np.int32)
del all_paths, all_labels, paths_tmp, labels_tmp
print(f'Train: {len(y_train)} | Val: {len(y_val)} | Test: {len(y_test)}')
print(f'Train balance: {sum(y_train)}/{len(y_train)} (1s/{len(y_train)})')

In [ ]:
# Precompute resized JPEG + ELA PNG cache for all splits
# Eliminates py_function dependency and ELA encoder mismatch
import os
from pathlib import Path
# Cache MUST be writable â€” dataset dir is often read-only (Kaggle/Colab mounts)
_cache_bases = [Path('/kaggle/working'), Path('/content'), Path.cwd()]
CACHE_ROOT = next((b for b in _cache_bases if b.exists() and os.access(b, os.W_OK)), Path.cwd())
CACHE_DIR = CACHE_ROOT / 'forgery_cache'
print(f'Cache dir: {CACHE_DIR}')

import shutil
_, _, free_bytes = shutil.disk_usage(CACHE_ROOT)
free_gb = free_bytes / (1024**3)
if free_gb < 5:
    raise RuntimeError(f'Only {free_gb:.1f}GB free, need at least 5GB for cache')
print(f'Free disk: {free_gb:.1f}GB')

from concurrent.futures import ThreadPoolExecutor, as_completed
import hashlib

def prepare_cache(paths, desc, chunk_size=2000):
    raw_cache, ela_cache = [], []
    def _proc(p):
        h = hashlib.md5(p.encode()).hexdigest()[:20]
        rp = CACHE_DIR / 'raw' / f'{h}.jpg'
        ep = CACHE_DIR / 'ela' / f'{h}.png'
        if not rp.exists() or not ep.exists():
            img = Image.open(p).convert('RGB')
            rp.parent.mkdir(parents=True, exist_ok=True)
            img.resize(IMG_SIZE, Image.LANCZOS).save(rp, 'JPEG', quality=95)
            ep.parent.mkdir(parents=True, exist_ok=True)
            compute_ela(p, quality=ELA_QUALITY, target_size=IMG_SIZE, pil_image=img).save(ep)
        return str(rp), str(ep)
    for start in range(0, len(paths), chunk_size):
        chunk = paths[start:start+chunk_size]
        with ThreadPoolExecutor(max_workers=4) as ex:
            futures = [ex.submit(_proc, p) for p in chunk]
            for f in as_completed(futures):
                rp, ep = f.result()
                raw_cache.append(rp)
                ela_cache.append(ep)
        gc.collect()
        print(f'  {desc}: {min(start+chunk_size, len(paths))}/{len(paths)}')
    return raw_cache, ela_cache
raw_train, ela_train = prepare_cache(paths_train, 'train')
del paths_train
gc.collect()
raw_val, ela_val = prepare_cache(paths_val, 'val')
del paths_val
gc.collect()
raw_test, ela_test = prepare_cache(paths_test, 'test')
# Keep paths_test for Grad-CAM visualization (list of strings, ~1MB)
gc.collect()
print('Cache ready.')


In [ ]:
def load_image(jpeg_path, png_path):
    raw = tf.image.decode_jpeg(tf.io.read_file(jpeg_path), channels=3)
    ela = tf.image.decode_png(tf.io.read_file(png_path), channels=3)
    raw.set_shape([*IMG_SIZE, 3])
    ela.set_shape([*IMG_SIZE, 3])
    return raw, ela

def random_jpeg(img, min_q=10, max_q=100):
    img_u8 = tf.cast(tf.clip_by_value(tf.cast(img, tf.int32), 0, 255), tf.uint8)
    return tf.cast(tf.image.random_jpeg_quality(img_u8, min_q, max_q), tf.float32)
def fixed_jpeg(img, quality=75):
    img_u8 = tf.cast(tf.clip_by_value(tf.cast(img, tf.int32), 0, 255), tf.uint8)
    return tf.cast(tf.io.decode_jpeg(tf.io.encode_jpeg(img_u8, quality=quality), channels=3), tf.float32)

def make_dual_ds(raw_paths, ela_paths, labels, training=False, tta=False, seed=SEED):
    augment = training or tta
    rot_layer = tf.keras.layers.RandomRotation(10/360, fill_mode='nearest', seed=seed) if augment else None
    zoom_layer = tf.keras.layers.RandomZoom(height_factor=(-0.1, 0.1), fill_mode='nearest', seed=seed) if augment else None
    ds = tf.data.Dataset.from_tensor_slices((raw_paths, ela_paths, labels))
    def _process(rp, ep, label):
        raw, ela = load_image(rp, ep)
        # JPEG augmentation on RAW only
        # ELA residuals encode compression-difference signal — must NOT be re-compressed
        if training:
            raw = random_jpeg(raw)
        else:
            raw = fixed_jpeg(raw)
        if training:
            # P2: color augs â€” RAW only (preserve ELA compression artifacts)
            raw = tf.image.random_brightness(raw, 0.15)
            raw = tf.clip_by_value(raw, 0, 255)
            raw = tf.image.random_contrast(raw, 0.7, 1.3)
            raw = tf.clip_by_value(raw, 0, 255)
            raw = tf.image.random_hue(raw / 255.0, 0.05) * 255.0
            raw = tf.image.random_saturation(raw / 255.0, 0.8, 1.2) * 255.0
            raw = tf.clip_by_value(raw, 0, 255)
        if augment:
            # P2/P4: geometric augs â€” shared via stacking (also for TTA)
            raw_ela = tf.stack([tf.cast(raw, tf.uint8), tf.cast(ela, tf.uint8)], axis=0)
            raw_ela = tf.image.random_flip_left_right(raw_ela, seed=seed)
            raw_ela = rot_layer(raw_ela)
            raw_ela = zoom_layer(raw_ela)
            raw, ela = tf.cast(raw_ela[0], tf.int32), tf.cast(raw_ela[1], tf.int32)
        raw = preprocess_input(tf.cast(raw, tf.float32))
        ela = preprocess_input(tf.cast(ela, tf.float32))
        return {'raw_input': raw, 'ela_input': ela}, tf.cast(label, tf.float32)
    if training:
        ds = ds.shuffle(min(len(labels), 2048), seed=seed)
    return ds.map(_process, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(1)

# P3: Self-Blended Images (SBI) augmentation — replaces CutMix
# Only apply SBI to authentic images (label=1) — simulates forgery on real images
# Apply rate 10% keeps batch class balance near 1.2:1 (tampered:authentic)
def sbi_batch(data, labels):
    raw, ela = data['raw_input'], data['ela_input']
    bs = tf.shape(raw)[0]
    h = w = 224
    xs = tf.cast(tf.range(w), tf.float32)[tf.newaxis, tf.newaxis, :]
    ys = tf.cast(tf.range(h), tf.float32)[tf.newaxis, :, tf.newaxis]
    cx = tf.random.uniform([bs, 1, 1], 0.0, tf.cast(w, tf.float32))
    cy = tf.random.uniform([bs, 1, 1], 0.0, tf.cast(h, tf.float32))
    rx = tf.random.uniform([bs, 1, 1], 0.05, 0.35) * tf.cast(w, tf.float32)
    ry = tf.random.uniform([bs, 1, 1], 0.05, 0.35) * tf.cast(h, tf.float32)
    angle = tf.random.uniform([bs, 1, 1], 0.0, 2 * math.pi)
    cos_a, sin_a = tf.cos(angle), tf.sin(angle)
    X_rot = (xs - cx) * cos_a + (ys - cy) * sin_a
    Y_rot = -(xs - cx) * sin_a + (ys - cy) * cos_a
    ellipse = (X_rot / rx)**2 + (Y_rot / ry)**2
    mask = tf.cast(ellipse <= 1.0, tf.float32)[..., tf.newaxis]
    is_auth = tf.cast(labels > 0.5, tf.float32)  # only apply to authentic
    apply = tf.cast(tf.random.uniform([bs, 1, 1, 1]) < 0.1, tf.float32) * is_auth[..., tf.newaxis, tf.newaxis]
    mask = mask * apply
    raw_blur = tf.nn.avg_pool2d(raw, 5, 1, 'SAME')
    ela_blur = tf.nn.avg_pool2d(ela, 5, 1, 'SAME')
    strength = tf.random.uniform([bs, 1, 1, 1], 0.5, 3.0)
    raw_source = tf.clip_by_value(raw + strength * (raw - raw_blur), -1.0, 1.0)
    ela_source = tf.clip_by_value(ela + strength * (ela - ela_blur), -1.0, 1.0)
    raw_out = raw * (1.0 - mask) + raw_source * mask
    ela_out = ela * (1.0 - mask) + ela_source * mask
    apply_flat = apply[:, 0, 0, 0]
    new_labels = labels * (1.0 - apply_flat)
    return {'raw_input': raw_out, 'ela_input': ela_out}, new_labels

train_ds = make_dual_ds(raw_train, ela_train, y_train, training=True)
val_ds = make_dual_ds(raw_val, ela_val, y_val)
test_ds = make_dual_ds(raw_test, ela_test, y_test)
del raw_train, ela_train, raw_val, ela_val
gc.collect()

# Added to training pipeline (Phase 2 only — Phase 1 uses clean train_ds)
train_phase2_ds = train_ds.map(sbi_batch, num_parallel_calls=tf.data.AUTOTUNE)
print('SBI enabled for Phase 2.')
print('Datasets ready.')

In [ ]:
# Dual-input EfficientNetB3 with cross-attention fusion (P1)
tf.keras.backend.clear_session()
gc.collect()
raw_in = layers.Input(shape=(224,224,3), name='raw_input')
ela_in = layers.Input(shape=(224,224,3), name='ela_input')

# Download weights once, load into two uniquely-named backbones
BASE_WT = 'https://storage.googleapis.com/keras-applications/efficientnetb3_notop.h5'
wt_path = tf.keras.utils.get_file('efficientnetb3_notop.h5', BASE_WT)

raw_bb = EfficientNetB3(include_top=False, weights=None, input_shape=(224,224,3), name='efficientnetb3_raw')
ela_bb = EfficientNetB3(include_top=False, weights=None, input_shape=(224,224,3), name='efficientnetb3_ela')
raw_bb.load_weights(wt_path)
ela_bb.load_weights(wt_path)
raw_bb.trainable = False
ela_bb.trainable = False

raw_feat = layers.GlobalAveragePooling2D()(raw_bb(raw_in))
ela_feat = layers.GlobalAveragePooling2D()(ela_bb(ela_in))

# P1: Cross-attention fusion â€” product (agreement) + difference (disagreement)
product = layers.Multiply()([raw_feat, ela_feat])
diff = layers.Subtract()([raw_feat, ela_feat])
fused = layers.Concatenate()([raw_feat, ela_feat, product, diff])
# D: Stronger regularisation â€” head has 262K params with cross-attention
fused = layers.Dropout(0.5)(fused)
fused = layers.Dense(512, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(4e-4))(fused)
fused = layers.Dropout(0.5)(fused)
fused = layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(4e-4))(fused)
fused = layers.Dropout(0.3)(fused)
out = layers.Dense(1, activation='sigmoid', name='forgery_prob')(fused)

model = Model(inputs=[raw_in, ela_in], outputs=out)
model.summary()

# Pre-build Grad-CAM sub-model (before training modifies graph)
raw_sub_gc = model.get_layer('efficientnetb3_raw')
_conv_name = next((l.name for l in reversed(raw_sub_gc.layers) if isinstance(l, layers.Conv2D)), None)
assert _conv_name is not None, 'No Conv2D for Grad-CAM'
gm = Model(model.inputs, [raw_sub_gc.get_layer(_conv_name).output, model.output])

In [ ]:
def calibrate_threshold(p_auth, y_true):
    best = {'thr': 0.5, 'score': -1, 'accuracy': 0, 'tampered_f1': 0, 'authentic_f1': 0}
    for thr in np.linspace(0.05, 0.95, 181):
        yp = (p_auth >= thr).astype(int)
        acc = np.mean(yp == y_true)
        tp_t = np.sum((y_true==0)&(yp==0)); fp_t = np.sum((y_true==1)&(yp==0)); fn_t = np.sum((y_true==0)&(yp==1))
        f1_t = 2*tp_t/(2*tp_t+fp_t+fn_t+1e-12)
        tp_a = np.sum((y_true==1)&(yp==1)); fp_a = np.sum((y_true==0)&(yp==1)); fn_a = np.sum((y_true==1)&(yp==0))
        f1_a = 2*tp_a/(2*tp_a+fp_a+fn_a+1e-12)
        if acc > best['score']:
            best = {'thr': float(thr), 'score': float(acc), 'accuracy': float(acc), 'tampered_f1': float(f1_t), 'authentic_f1': float(f1_a)}
    return best

In [ ]:
# Phase 1: frozen backbones
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

cbs = [
    EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_phase1_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=1),
]
print('PHASE 1: Frozen backbones')
h1 = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=cbs, verbose=1)

In [ ]:
# Phase 2: Progressive unfreezing (3 stages) + Focal Loss + SWA
def focal_loss(gamma=2., alpha=0.5):
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1-1e-7)
        pt = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        ce = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        return tf.reduce_mean(alpha_t * (1 - pt)**gamma * ce)
    return loss

class SWACallback(tf.keras.callbacks.Callback):
    def __init__(self, start_epoch=1):
        super().__init__()
        self.start_epoch = start_epoch
        self.avg_weights = None
        self.n = 0
    def on_epoch_end(self, epoch, logs=None):
        if epoch + 1 >= self.start_epoch:
            w = self.model.get_weights()
            if self.avg_weights is None:
                self.avg_weights = [x.copy() for x in w]
            else:
                n = self.n
                self.avg_weights = [
                    a * n / (n + 1) + x / (n + 1)
                    for a, x in zip(self.avg_weights, w)
                ]
            self.n += 1
    def on_train_end(self, logs=None):
        if self.avg_weights is not None:
            self.model.set_weights(self.avg_weights)
            print(f'SWA: averaged {self.n} checkpoints')

# Hard guarantee against overfitting: stop if train-val accuracy gap exceeds ceiling
class OverfitGuard(tf.keras.callbacks.Callback):
    def __init__(self, max_gap=0.04, patience=3):
        super().__init__()
        self.max_gap = max_gap
        self.patience = patience
        self.bad_epochs = 0
        self.stopped_epoch = 0
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        tr = logs.get('accuracy')
        va = logs.get('val_accuracy')
        if tr is None or va is None:
            return
        gap = tr - va
        if gap > self.max_gap:
            self.bad_epochs += 1
            print(f'OverfitGuard: train-val gap {gap*100:.1f}% > {self.max_gap*100:.0f}% (epoch {epoch+1})')
            if self.bad_epochs >= self.patience:
                self.stopped_epoch = epoch
                self.model.stop_training = True
        else:
            self.bad_epochs = 0
    def on_train_end(self, logs=None):
        if self.stopped_epoch > 0:
            print(f'OverfitGuard: stopped at epoch {self.stopped_epoch+1} (gap > {self.max_gap*100:.0f}% for {self.patience} epochs)')

from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.metrics import AUC

# B: Progressive unfreezing â€” 3 stages, each with lower LR and more blocks
BLOCKS_P2 = [[7], [5, 6, 7], [3, 4, 5, 6, 7]]
LR_P2 = [1e-5, 5e-6, 2e-6]
EPOCHS_P2 = [5, 7, 15]
NAMES_P2 = ['block7 (1e-5)', 'blocks5-7 (5e-6)', 'blocks3-7 (2e-6)']

for blocks, lr, epochs, name in zip(BLOCKS_P2, LR_P2, EPOCHS_P2, NAMES_P2):
    for bb in [raw_bb, ela_bb]:
        bb.trainable = True
        for l in bb.layers:
            if isinstance(l, tf.keras.layers.BatchNormalization):
                l.trainable = (name == NAMES_P2[-1])
            elif any(f'block{n}' in l.name for n in blocks) or 'top_conv' in l.name:
                l.trainable = True
            else:
                l.trainable = False

    steps = len(y_train) // BATCH_SIZE
    lr_schedule = CosineDecay(lr, decay_steps=epochs * steps, alpha=0.02)
    model.compile(optimizer=AdamW(lr_schedule, weight_decay=1e-4, global_clipnorm=1.0),
                  loss=focal_loss(gamma=2., alpha=0.5),
                  metrics=['accuracy', AUC(name='auc')])

    cbs = [OverfitGuard(max_gap=0.04, patience=3)]
    if name == NAMES_P2[-1]:
        cbs += [
            EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True, verbose=1),
            ModelCheckpoint('best_forgery_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=1),
            SWACallback(start_epoch=1),
        ]

    print(f'PHASE 2: {name}')
    model.fit(train_phase2_ds, validation_data=val_ds, epochs=epochs, callbacks=cbs, verbose=1)

In [ ]:
# Evaluate + save (with TTA)
val_preds = model.predict(val_ds, verbose=0).ravel()
best = calibrate_threshold(val_preds, y_val)

# P4: Test-Time Augmentation â€” average 5 predictions with different seeds
tta_preds = []
for i in range(5):
    tta_ds = make_dual_ds(raw_test, ela_test, y_test, tta=True, seed=SEED + i + 1)
    tta_preds.append(model.predict(tta_ds, verbose=0).ravel())
    del tta_ds
test_preds = np.mean(tta_preds, axis=0)
print(f'TTA: averaged {len(tta_preds)} predictions')
y_pred = (test_preds >= best['thr']).astype(int)

acc = float(np.mean(y_pred == y_test))
tamp = float(np.mean(y_pred[y_test==0]==0)) if np.any(y_test==0) else 0
auth = float(np.mean(y_pred[y_test==1]==1)) if np.any(y_test==1) else 0
cm = confusion_matrix(y_test, y_pred, labels=[0,1])

model.save('best_forgery_model.keras')
with open('best_threshold.json', 'w') as f:
    json.dump({'threshold_authentic': best['thr']}, f)

print(f'\n=== RESULTS ===')
print(f'Threshold: {best["thr"]:.3f}')
print(f'Test Accuracy: {acc*100:.2f}%')
print(f'Tampered: {tamp*100:.2f}% | Authentic: {auth*100:.2f}%')
print(f'Tampered F1: {best["tampered_f1"]:.4f} | Authentic F1: {best["authentic_f1"]:.4f}')
print(f'\nConfusion Matrix [rows=true, cols=pred]:\n{cm}')

In [ ]:
# Rebuild Grad-CAM model after training (graph references changed by SWA)
raw_sub_gc = model.get_layer('efficientnetb3_raw')
_conv_name = next((l.name for l in reversed(raw_sub_gc.layers) if isinstance(l, layers.Conv2D)), None)
gm = Model(model.inputs, [raw_sub_gc.get_layer(_conv_name).output, model.output])
print('Grad-CAM sub-model rebuilt.')

In [ ]:
# Grad-CAM demo (loads 2 test images on-demand)
# gm was rebuilt above after training â€” safe to use here
def gradcam(raw01, ela01):
    r = tf.convert_to_tensor(raw01[None,...])
    e = tf.convert_to_tensor(ela01[None,...])
    with tf.GradientTape() as tape:
        co, pr = gm([r, e], training=False)
        s = pr[:,0]
    g = tape.gradient(s, co)
    cam = tf.reduce_sum(co[0] * tf.reduce_mean(g, axis=(0,1,2)), axis=-1)
    cam = tf.maximum(cam, 0)
    return (cam / (tf.reduce_max(cam)+1e-8)).numpy()

def overlay(img01, cam, a=0.4):
    h, w = img01.shape[:2]
    c = cv2.applyColorMap(np.uint8(255*cv2.resize(cam,(w,h))), cv2.COLORMAP_JET)[:,:,::-1]/255.0
    return np.clip((1-a)*img01 + a*c, 0, 1)

# Load 2 test images from paths
gc_paths = [paths_test[0], paths_test[len(paths_test)//2]]
gc_ys = [y_test[0], y_test[len(y_test)//2]]
raw_01, ela_01 = [], []
for p in gc_paths:
    img = Image.open(p).convert('RGB').resize(IMG_SIZE, Image.LANCZOS)
    raw_01.append(np.asarray(img, dtype=np.float32) / 255.0)
    ela_01.append(np.asarray(compute_ela(p, quality=ELA_QUALITY, target_size=IMG_SIZE), dtype=np.float32) / 255.0)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i in range(2):
    r, e = raw_01[i], ela_01[i]
    p = float(model.predict_on_batch([r[None,...], e[None,...]])[0,0])
    lbl = 'Auth' if p >= best['thr'] else 'Fake'
    true = 'Auth' if gc_ys[i]==1 else 'Fake'
    cam = gradcam(preprocess_input(r*255), preprocess_input(e*255))
    axes[i,0].imshow(r); axes[i,0].set_title(f'Raw ({true})')
    axes[i,1].imshow(e); axes[i,1].set_title('ELA')
    axes[i,2].imshow(overlay(r, cam)); axes[i,2].set_title(f'{lbl} {p:.3f}')
    for ax in axes[i]: ax.axis('off')
plt.tight_layout()
plt.show()
del raw_01, ela_01, gc_paths, gc_ys  # free memory

In [ ]:
# Copy model to Drive (optional â€” run this manually after training)
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        import shutil
        for f in ['best_forgery_model.keras', 'best_phase1_model.keras', 'best_threshold.json']:
            if Path(f).exists():
                shutil.copy(f, '/content/drive/MyDrive/')
                print(f'Copied {f} to Drive')
    except Exception as e:
        print(f'Drive copy failed (non-fatal): {e}')